# Task 5 — RAG System using LangChain

Answers questions over a PDF or Excel knowledge base using Retrieval-Augmented
Generation (RAG): a **retriever** finds the most relevant chunks and a
**generator** LLM composes the answer from them.

## Objectives checklist
- [x] Load a PDF as knowledge base
- [x] Load an Excel sheet as knowledge base
- [x] Split documents into chunks
- [x] Build a retriever (embeddings + FAISS vector index)
- [x] Build a generator (LLM that answers from retrieved context)
- [x] Wire retriever + generator into one RAG system
- [x] Automated tests proving retrieval and generation work correctly

This notebook is fully self-contained: it generates its own sample PDF and
Excel knowledge base, builds the RAG pipeline, runs a demo, and runs an
offline automated test suite -- all in one file.

## About `USE_LIVE_MODEL`
Set the flag below to `True` to download and use real models from Hugging
Face (`sentence-transformers/all-MiniLM-L6-v2` for embeddings and
`google/flan-t5-base` for generation) -- this requires internet access.
Left `False`, the notebook uses lightweight deterministic offline
stand-ins so every cell can be verified without downloading anything,
which is what was used to produce the outputs saved in this notebook.

In [1]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-core \
    faiss-cpu sentence-transformers transformers torch pypdf openpyxl pandas reportlab

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [2]:
USE_LIVE_MODEL = False  # Set True to use real Hugging Face embeddings + LLM (requires internet)


## 2. Generate the sample knowledge base (PDF + Excel)

So this notebook needs no external files.

In [3]:
import os

os.makedirs("sample_data", exist_ok=True)

# --- Sample Excel FAQ sheet ---
import openpyxl

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "FAQ"
ws.append(["Question", "Answer"])
faq_rows = [
    ("What are the company working hours?", "The company operates from 9 AM to 6 PM, Sunday to Thursday."),
    ("What is the refund policy?", "Refunds are processed within 14 days of purchase if the product is unused."),
    ("How can I contact support?", "Support can be reached via email at support@talent-360.me or through the live chat on the website."),
    ("What is the warranty period?", "All products come with a 12-month manufacturer warranty."),
    ("Do you offer international shipping?", "Yes, we ship to over 30 countries with delivery times of 5-10 business days."),
]
for row in faq_rows:
    ws.append(row)
wb.save("sample_data/company_faq.xlsx")

# --- Sample PDF handbook ---
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

c = canvas.Canvas("sample_data/company_handbook.pdf", pagesize=letter)
width, height = letter
pages = [
    ["Talent360 Employee Handbook", "",
     "Section 1: Working Hours",
     "Employees work from 9 AM to 6 PM, Sunday to Thursday.",
     "Remote work is permitted up to two days per week with manager approval."],
    ["Section 2: Leave Policy", "",
     "Employees are entitled to 21 paid vacation days per year.",
     "Sick leave requires a medical certificate for absences longer than 2 days."],
    ["Section 3: Code of Conduct", "",
     "Employees must treat colleagues and clients with respect and professionalism.",
     "Confidential company information must not be shared outside the organization."],
]
for page_lines in pages:
    y = height - 100
    for line in page_lines:
        c.drawString(72, y, line)
        y -= 24
    c.showPage()
c.save()

print("Sample knowledge base created: sample_data/company_faq.xlsx, sample_data/company_handbook.pdf")

Sample knowledge base created: sample_data/company_faq.xlsx, sample_data/company_handbook.pdf


## 3. Document loaders

One loader for PDF (`PyPDFLoader`), one custom loader for Excel (one `Document` per row -- avoids the heavy `unstructured` dependency).

In [4]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_excel(file_path: str) -> list[Document]:
    """Loads every sheet of an Excel workbook into one Document per row."""
    sheets = pd.read_excel(file_path, sheet_name=None)
    documents = []
    for sheet_name, df in sheets.items():
        for row_idx, row in df.iterrows():
            text = "; ".join(f"{col}: {val}" for col, val in row.items() if pd.notna(val))
            if text:
                documents.append(
                    Document(page_content=text, metadata={"source": file_path, "sheet": sheet_name, "row": int(row_idx)})
                )
    return documents


def load_documents(file_path: str) -> list[Document]:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return PyPDFLoader(file_path).load()
    if ext in (".xlsx", ".xls"):
        return load_excel(file_path)
    raise ValueError(f"Unsupported file type: {ext}. Use a .pdf or .xlsx file.")


def split_documents(documents: list[Document], chunk_size: int = 500, chunk_overlap: int = 50) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(documents)


print("Loaders ready.")

/tmp/ipykernel_2212/240569251.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaders ready.


## 4. Retriever

Embeds chunks and stores them in a FAISS vector index. `FakeEmbeddings` is a deterministic bag-of-words embedding used only when `USE_LIVE_MODEL = False`, so retrieval can be demonstrated and tested without downloading a model.

In [5]:
import hashlib


class FakeEmbeddings(Embeddings):
    """Deterministic offline stand-in for a real embedding model."""

    def _embed(self, text: str):
        vector = [0.0] * 32
        for word in text.lower().split():
            idx = int(hashlib.md5(word.encode()).hexdigest(), 16) % 32
            vector[idx] += 1.0
        return vector

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


def get_embeddings():
    if USE_LIVE_MODEL:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return FakeEmbeddings()


def build_vector_store(chunks: list[Document]) -> FAISS:
    return FAISS.from_documents(chunks, get_embeddings())


print(f"Retriever configured (USE_LIVE_MODEL={USE_LIVE_MODEL}).")

Retriever configured (USE_LIVE_MODEL=False).


## 5. Generator

When `USE_LIVE_MODEL = True`, this loads `google/flan-t5-base` from Hugging Face. Otherwise it uses a simple offline extractive fallback (returns the most relevant retrieved sentence) so the pipeline can be demonstrated end to end without a model download.

In [6]:
PROMPT_TEMPLATE = (
    "Answer the question using only the context below. If the answer is not "
    "in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)


def offline_generator(prompt: str) -> str:
    """Extractive fallback: returns the retrieved context (offline demo only).

    Uses rsplit on the last "\n\nQuestion:" marker (the one the prompt
    template appends after the context) since retrieved chunks can
    themselves legitimately contain the word "Question:".
    """
    context = prompt.split("Context:\n", 1)[1].rsplit("\n\nQuestion:", 1)[0].strip()
    return f"(offline demo answer, based on retrieved context) {context}"


def get_generator():
    if USE_LIVE_MODEL:
        from transformers import pipeline
        pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=256)
        return lambda p: pipe(p)[0]["generated_text"].strip()
    return offline_generator


class RAGSystem:
    """Ties the retriever (vector store) and generator (LLM) together."""

    def __init__(self, vector_store: FAISS, generator, top_k: int = 3):
        self.vector_store = vector_store
        self.generator = generator
        self.top_k = top_k

    @classmethod
    def from_file(cls, file_path: str, top_k: int = 3) -> "RAGSystem":
        documents = load_documents(file_path)
        chunks = split_documents(documents)
        vector_store = build_vector_store(chunks)
        return cls(vector_store, get_generator(), top_k=top_k)

    def retrieve(self, question: str) -> list[Document]:
        return self.vector_store.similarity_search(question, k=self.top_k)

    def build_prompt(self, question: str, retrieved_docs: list[Document]) -> str:
        context = "\n\n".join(doc.page_content for doc in retrieved_docs)
        return PROMPT_TEMPLATE.format(context=context, question=question)

    def answer(self, question: str):
        retrieved_docs = self.retrieve(question)
        prompt = self.build_prompt(question, retrieved_docs)
        return self.generator(prompt), retrieved_docs


print("RAGSystem ready.")

RAGSystem ready.


## 6. Demo run

Builds the RAG system from the sample Excel FAQ and asks a few questions.

In [7]:
rag = RAGSystem.from_file("sample_data/company_faq.xlsx")

demo_questions = [
    "What is the refund policy?",
    "What are the working hours?",
    "Do you ship internationally?",
]

for question in demo_questions:
    answer, sources = rag.answer(question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("Sources:")
    for doc in sources:
        print(f"  - {doc.page_content}")
    print()

Q: What is the refund policy?
A: (offline demo answer, based on retrieved context) Question: What is the warranty period?; Answer: All products come with a 12-month manufacturer warranty.

Question: What is the refund policy?; Answer: Refunds are processed within 14 days of purchase if the product is unused.

Question: How can I contact support?; Answer: Support can be reached via email at support@talent-360.me or through the live chat on the website.
Sources:
  - Question: What is the warranty period?; Answer: All products come with a 12-month manufacturer warranty.
  - Question: What is the refund policy?; Answer: Refunds are processed within 14 days of purchase if the product is unused.
  - Question: How can I contact support?; Answer: Support can be reached via email at support@talent-360.me or through the live chat on the website.

Q: What are the working hours?
A: (offline demo answer, based on retrieved context) Question: What is the warranty period?; Answer: All products come w

## 7. Automated tests (offline)

Proves the retriever actually surfaces relevant chunks, and that loading/splitting/error-handling work -- without needing a model download.

In [8]:
def test_load_and_split_excel():
    documents = load_documents("sample_data/company_faq.xlsx")
    assert len(documents) > 0
    assert any("refund" in d.page_content.lower() for d in documents)
    chunks = split_documents(documents)
    assert len(chunks) >= len(documents)
    print(f"PASS test_load_and_split_excel ({len(documents)} rows -> {len(chunks)} chunks)")


def test_load_pdf():
    documents = load_documents("sample_data/company_handbook.pdf")
    assert len(documents) > 0
    print(f"PASS test_load_pdf ({len(documents)} pages)")


def test_retriever_finds_relevant_chunk():
    documents = load_documents("sample_data/company_faq.xlsx")
    chunks = split_documents(documents)
    vector_store = build_vector_store(chunks)
    rag_local = RAGSystem(vector_store, offline_generator, top_k=2)
    retrieved = rag_local.retrieve("What is the refund policy?")
    assert any("refund" in d.page_content.lower() for d in retrieved)
    print("PASS test_retriever_finds_relevant_chunk")


def test_answer_uses_retrieved_context():
    documents = load_documents("sample_data/company_faq.xlsx")
    chunks = split_documents(documents)
    vector_store = build_vector_store(chunks)
    rag_local = RAGSystem(vector_store, offline_generator, top_k=2)
    answer, sources = rag_local.answer("What are the working hours?")
    assert "offline demo answer" in answer
    assert len(sources) == 2
    print("PASS test_answer_uses_retrieved_context")


def test_unsupported_file_type_raises():
    try:
        load_documents("notes.txt")
    except ValueError:
        print("PASS test_unsupported_file_type_raises")
        return
    raise AssertionError("Expected ValueError for unsupported file type")


test_load_and_split_excel()
test_load_pdf()
test_retriever_finds_relevant_chunk()
test_answer_uses_retrieved_context()
test_unsupported_file_type_raises()
print("\nAll RAG pipeline tests passed.")

PASS test_load_and_split_excel (5 rows -> 5 chunks)
PASS test_load_pdf (3 pages)
PASS test_retriever_finds_relevant_chunk
PASS test_answer_uses_retrieved_context
PASS test_unsupported_file_type_raises

All RAG pipeline tests passed.


## 8. Interactive mode (optional)

Run this cell in a real Jupyter session to ask your own questions. It exits gracefully if there is no input available (e.g. when run non-interactively).

In [9]:
try:
    while True:
        question = input("Question (or 'exit'): ").strip()
        if not question or question.lower() in ("exit", "quit"):
            break
        answer, sources = rag.answer(question)
        print(f"Answer: {answer}\n")
except Exception:
    print("(No interactive input available -- skipping interactive mode.)")

(No interactive input available -- skipping interactive mode.)


## Notes
- Swap `get_generator()` for an API-based LLM (e.g. `ChatOpenAI`, `ChatAnthropic`) if you have API access -- the retrieval logic stays unchanged.
- Package versions verified working: `langchain` 1.3.x, `langchain-community` 0.4.x, `faiss-cpu` 1.15.x, `transformers` 5.x, `torch` 2.13.x. The legacy `RetrievalQA` chain from LangChain 0.x was intentionally **not** used since it was removed in `langchain` 1.0 -- this notebook builds the retriever+generator steps explicitly instead.
